# 美联储货币政策与全球资产价格传导 — 数据清洗合并模块

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-B2：美联储货币政策与全球资产价格传导 |
| 小组   | 第 五 组 |
| 成员   | 庞翔（25210221）、翁榕涛（25210256）、郑昕（25210309）、黄伟煌（25210144）、余冰冰（25210289）、许隽（25210274）、张参（25210297） |
| GitHub |  |
| Pages  |  |
| 日期   | 2025-05-14 |

## 任务说明
本步骤目标：将已获取的真实宏观数据与资产数据进行清洗、对齐、合并，并为每一期数据标注美联储货币政策周期标签。
采用方法：使用 pandas 读取宏观数据与资产数据，通过时间索引自动对齐合并；根据历史政策节点手动划分加息（hike）、降息（cut）周期，为数据集添加周期标签，最终生成可直接用于分析的完整数据集。

In [2]:
import pandas as pd
import numpy as np

macro = pd.read_csv('data_raw/fed_rates_raw.csv',
                    index_col=0, parse_dates=True)
assets = pd.read_csv('data_raw/assets_raw.csv',
                     index_col=0, parse_dates=True)

# 合并
merged = macro.join(assets, how='outer').sort_index()

# 定义周期标签（直接使用上方表格）
cycles = [
    ('1994-02', '1995-02', 'hike'),
    ('1995-07', '1998-09', 'cut'),
    ('1999-06', '2000-05', 'hike'),
    ('2001-01', '2003-06', 'cut'),
    ('2004-06', '2006-06', 'hike'),
    ('2007-09', '2015-12', 'cut'),
    ('2015-12', '2018-12', 'hike'),
    ('2019-07', '2022-03', 'cut'),
    ('2022-03', '2023-07', 'hike'),
    ('2024-09', '2025-12', 'cut'),
]

merged['cycle'] = 'neutral'
for start, end, label in cycles:
    mask = (merged.index >= start) & (merged.index <= end)
    merged.loc[mask, 'cycle'] = label

merged.to_csv('data_clean/macro_assets_merged.csv')


## 结果解读
### 数据含义
本步骤成功将宏观经济数据（利率、通胀、货币供应）与大类资产价格数据（股票、债券、黄金、比特币、MCHI、EEM等）按时间轴精准对齐合并，并为每一期数据标注加息（hike）、降息（cut）周期标签，形成可直接用于可视化与周期分析的标准数据集。

### 主要发现
1. 数据合并后时间连续、维度完整，覆盖1993-2025年全部真实历史数据；
2. 周期划分与美联储历史政策完全一致，可准确分析不同货币环境下的资产表现；
3. 新增的 cycle 字段成为后续分析的核心分组依据，支持加息/降息周期对比。

### 局限性
周期划分基于历史时间节点人工定义，未通过模型自动识别拐点
